# De-repression score — pseudobulk pipeline (v2)

Fixes the depth-confound present in v1:
- **v1**: AUCell scored per cell on log1p-norm → 90th pct per donor → confounded by UMI depth
- **v2**: Sum raw counts per donor → CPM → mean log-CPM of gene set → depth-robust by construction

Gene set is still derived from BICAN healthy donors using `find_derepressed_genes`.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from dotenv import load_dotenv

load_dotenv()

True

## Configuration

In [2]:
CT_FOR_DEG_VARIABLE = "ct_for_deg"
SAMPLE_VARIABLE     = "donor_id"
CONTRAST_VARIABLE   = "condition"
CONTRAST_BASELINE   = "Control"
CONTRAST_STIM       = "XDP"
AGE                 = "age_of_death"
ZONE_COL            = "zone"
LIBRARY_SIZE_COL    = "total_counts"

ZONE             = "4"
MIN_CELLS        = 10
AGE_BUFFER       = 5    # years of tolerance around XDP age range
MIN_HEALTHY      = 3    # min healthy donors to keep per age bin
CACHE_DIR        = "/home/gdallagl/myworkdir/XDP/data/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

## Helper functions

In [3]:
def merge_adata(adatas):
    """Concatenate AnnData objects keeping only common genes."""
    common = set(adatas[0].var_names)
    for a in adatas[1:]:
        common &= set(a.var_names)
    common = sorted(common)
    adatas = [a[:, common].copy() for a in adatas]
    merged = adatas[0].concatenate(adatas[1:], batch_key=None, index_unique=None)
    merged.var = adatas[0].var.loc[common]
    return merged


def filter_low_cell_donors(adata, sample_col, min_cells):
    counts = adata.obs[sample_col].value_counts()
    keep   = counts[counts >= min_cells].index
    return adata[adata.obs[sample_col].isin(keep)].copy()


def downsample_healthy(donor_meta, age_col, sample_col, age_min, age_max,
                       xdp_per_decade, age_buffer=AGE_BUFFER,
                       min_keep=MIN_HEALTHY, seed=42):
    """Keep healthy donors within [age_min-buffer, age_max+buffer].
    Per 10-year bin keep at most max(n_xdp, min_keep) donors."""
    rng  = np.random.RandomState(seed)
    meta = donor_meta.copy()
    meta[age_col] = meta[age_col].astype(float)

    in_range = meta[
        (meta[age_col] >= age_min - age_buffer) &
        (meta[age_col] <= age_max + age_buffer)
    ].copy()
    in_range["decade"] = (in_range[age_col] // 10 * 10).astype(int)

    keep_ids = []
    for decade, grp in in_range.groupby("decade"):
        n_xdp = xdp_per_decade.get(decade, 0)
        if n_xdp == 0:
            continue
        n_target = max(n_xdp, min_keep)
        if len(grp) <= n_target:
            keep_ids.extend(grp[sample_col].tolist())
        else:
            keep_ids.extend(
                rng.choice(grp[sample_col].values, size=n_target, replace=False).tolist()
            )
    return set(keep_ids)

## Gene-set detection function (runs on BICAN pseudobulk)

In [4]:
def find_derepressed_genes(
    adata,
    age_col,
    donor_col,
    counts_col,
    other_covariate_cols=None,
    threshold=0.02,
    min_donor_cells=10,
    n_perm=1000,
    seed=42,
    fdr=0.05,
):
    """
    Identify genes that are:
      1. Nearly silent in ALL donors (detection rate < threshold)
      2. Show increasing detection with age (permutation test, FWL residualisation)

    Returns a DataFrame with one row per gene, sorted by p-value.
    """
    import scipy.stats as stats

    rng = np.random.RandomState(seed)
    if other_covariate_cols is None:
        other_covariate_cols = []

    # ── per-donor stats ────────────────────────────────────────────────────────
    donors = adata.obs[donor_col].unique()
    records = []
    for d in donors:
        cells = adata[adata.obs[donor_col] == d]
        if cells.n_obs < min_donor_cells:
            continue
        X = cells.layers["counts"]
        if scipy.sparse.issparse(X):
            X = X.toarray()
        det = (X > 0).mean(axis=0)          # detection rate per gene
        obs = cells.obs.iloc[0]
        row = {donor_col: d, age_col: float(obs[age_col]), "det": det}
        for c in other_covariate_cols:
            row[c] = obs[c]
        records.append(row)

    ages = np.array([r[age_col] for r in records])
    det_matrix = np.stack([r["det"] for r in records])   # donors × genes

    # ── candidate genes: silent in all donors ──────────────────────────────────
    max_det = det_matrix.max(axis=0)
    candidates = np.where(max_det < threshold)[0]
    det_cand   = det_matrix[:, candidates]

    # ── FWL residualisation of age ─────────────────────────────────────────────
    cov_cols = [np.ones(len(records))]
    for c in other_covariate_cols:
        vals = np.array([r[c] for r in records], dtype=float)
        cov_cols.append(vals)
    cov_matrix  = np.column_stack(cov_cols)
    hat         = cov_matrix @ np.linalg.pinv(cov_matrix)
    age_resid   = ages - hat @ ages
    det_resid   = det_cand - hat @ det_cand

    # ── observed Pearson r (age → detection) ──────────────────────────────────
    obs_r = np.array([
        stats.pearsonr(age_resid, det_resid[:, j])[0]
        for j in range(det_cand.shape[1])
    ])

    # ── permutation test (one-sided: r > 0) ───────────────────────────────────
    perm_counts = np.zeros(len(candidates))
    for _ in range(n_perm):
        age_perm = rng.permutation(age_resid)
        perm_r   = np.array([
            stats.pearsonr(age_perm, det_resid[:, j])[0]
            for j in range(det_cand.shape[1])
        ])
        perm_counts += (perm_r >= obs_r)

    pvals = (perm_counts + 1) / (n_perm + 1)
    _, qvals, _, _ = multipletests(pvals, method="fdr_bh")

    gene_names = adata.var_names[candidates]
    results = pd.DataFrame({
        "gene":        gene_names,
        "pearson_r":   obs_r,
        "pval":        pvals,
        "qval":        qvals,
        "max_det":     max_det[candidates],
        "significant": qvals < fdr,
    }).set_index("gene").sort_values("pval")

    print(f"Candidates (det < {threshold}): {len(candidates)} genes")
    print(f"Significant (FDR < {fdr}):      {results['significant'].sum()} genes")
    return results

## Load data

In [5]:
ADATA_PATH = "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/Striatum/Striatum_combined_QC_mmc_ct-cured_zoned.h5ad"
adata = sc.read_h5ad(ADATA_PATH, backed="r")
adata.var.index = adata.var["gene_symbol"]
adata.obs[CT_FOR_DEG_VARIABLE] = adata.obs[CT_FOR_DEG_VARIABLE].str.replace(" ", "_")
print(adata)

AnnData object with n_obs × n_vars = 231080 × 38601 backed at '/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/Striatum/Striatum_combined_QC_mmc_ct-cured_zoned.h5ad'
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'barcode', 'bcl', 'rna_index', 'library', 'library__barcode', 'frac_mito', 'mol_info_nUMI', 'mol_info_nRead', 'frac_intronic', 'donor_id', 'vireo_prob_max', 'vireo_prob_doublet', 'vireo_n_vars', 'vireo_best_singlet', 'vireo_best_doublet', 'vireo_doublet_logLikRatio', 'dropsift_frac_contamination', 'dropsift_training_label_is_cell', 'dropsift_empty_gene_module_score', 'dropsift_is_cell', 'dropsift_is_cell_prob', 'cell_class', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'tissue', 'broad_original_cell_type', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes

In [ ]:
BICAN_PATH = "/home/gdallagl/myworkdir/XDP/data/BICAN/STR_D1_Matrix/d1_cleaner_v3_raw_dv_wm_labeled.h5ad"
bican = sc.read_h5ad(BICAN_PATH)

# Harmonise column names
bican.obs.rename(columns={
    "merge_step_6" : ZONE_COL,
    "donor_id"     : SAMPLE_VARIABLE,
    "age"          : AGE,
    "n_counts"     : LIBRARY_SIZE_COL,
    "imputed_sex"  : "sex",
}, inplace=True, errors="ignore")
bican.obs[CONTRAST_VARIABLE] = CONTRAST_BASELINE
print(bican)

AnnData object with n_obs × n_vars = 206571 × 38100
    obs: 'prefix', 'cell_barcode', 'expression_doublet', 'num_genic_reads', 'num_transcripts', 'num_genes', 'num_retained_transcripts', 'pct_coding', 'pct_utr', 'pct_intergenic', 'pct_intronic', 'pct_mt', 'frac_contamination', 'donor_external_id', 'biobank', 'cohort', 'age', 'race', 'pmi_hr', 'imputed_sex', 'eur_adj', 'afr_adj', 'nat_adj', 'eas_adj', 'sas_adj', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'toxicology_group', 'toxicology_report_complete', 'toxicology_compounds_detected', 'hbcac_status', 'experiment', 'scpred_cortex_class', 'scpred_cortex_class_max_prob', 'scpred_cortex_gaba_sub_class', 'scpred_cortex_gaba_sub_class_max_prob', 'scpred_cortex_glut_sub_class', 'scpred_cortex_glut_sub_class_max_prob', 'scpred_caudate_class', 'scpred_caudate_class_max_prob', 'scpred_caudate_spn_di', 'scpred_caudate_spn_di_max_prob', 'scpred_caudate_spn_en', 'scpred_caudate_spn_en_max_prob', 'scpred_caudate_spn_mp', 'scpred_caudate_spn_mp_max_prob', '

## Subset to zone 4 (Matrix D1)

In [10]:
adata_zone = adata[adata.obs[ZONE_COL] == ZONE].to_memory().copy()
bican_zone = bican[bican.obs[ZONE_COL] == ZONE].copy()

adata_zone = filter_low_cell_donors(adata_zone, SAMPLE_VARIABLE, MIN_CELLS)
bican_zone = filter_low_cell_donors(bican_zone, SAMPLE_VARIABLE, MIN_CELLS)

print(f"NucSeq zone {ZONE}: {adata_zone.n_obs} cells, "
      f"{adata_zone.obs[SAMPLE_VARIABLE].nunique()} donors")
print(f"BICAN  zone {ZONE}: {bican_zone.n_obs} cells, "
      f"{bican_zone.obs[SAMPLE_VARIABLE].nunique()} donors")

NucSeq zone 4: 9404 cells, 47 donors
BICAN  zone 4: 0 cells, 0 donors


## Age distribution (before matching)

In [ ]:
def plot_age_dist(adata_z, bican_z, title=""):
    df_b = bican_z.obs.drop_duplicates(SAMPLE_VARIABLE).copy()
    df_n = adata_z.obs.drop_duplicates(SAMPLE_VARIABLE).copy()
    df_b["source"] = "BICAN (healthy)"
    df_n["source"] = df_n[CONTRAST_VARIABLE].map({
        CONTRAST_BASELINE: "NucSeq Control",
        CONTRAST_STIM:     "NucSeq XDP",
    })
    df = pd.concat([df_b[[AGE, "source"]], df_n[[AGE, "source"]]])
    df[AGE] = df[AGE].astype(float).dropna()
    df["age_bin"] = (df[AGE] // 5 * 5).astype(int).astype(str) + "–" + (df[AGE] // 5 * 5 + 5).astype(int).astype(str)
    order = sorted(df["age_bin"].dropna().unique(), key=lambda x: int(x.split("–")[0]))
    palette = {"BICAN (healthy)": "seagreen", "NucSeq XDP": "tomato", "NucSeq Control": "steelblue"}
    fig, ax = plt.subplots(figsize=(11, 3))
    sns.countplot(data=df.dropna(subset=["age_bin"]), x="age_bin", hue="source",
                  order=order, palette=palette, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Age bin")
    ax.set_ylabel("Donors")
    sns.despine()
    plt.tight_layout()
    plt.show()

plot_age_dist(adata_zone, bican_zone, title="Age distribution — before matching")

## Age-matching: restrict healthy donors to XDP age range

In [ ]:
xdp_meta = (
    adata_zone.obs[adata_zone.obs[CONTRAST_VARIABLE] == CONTRAST_STIM]
    .drop_duplicates(SAMPLE_VARIABLE)[[SAMPLE_VARIABLE, AGE]]
    .assign(**{AGE: lambda df: df[AGE].astype(float)})
)
xdp_age_min = xdp_meta[AGE].min()
xdp_age_max = xdp_meta[AGE].max()
xdp_meta["decade"] = (xdp_meta[AGE] // 10 * 10).astype(int)
xdp_per_decade = xdp_meta.groupby("decade").size()

print(f"XDP age range: {xdp_age_min:.0f}–{xdp_age_max:.0f}  ({len(xdp_meta)} donors)")
print("XDP donors per decade:\n", xdp_per_decade)

# Downsample NucSeq Controls
ctrl_meta = (
    adata_zone.obs[adata_zone.obs[CONTRAST_VARIABLE] == CONTRAST_BASELINE]
    .drop_duplicates(SAMPLE_VARIABLE)[[SAMPLE_VARIABLE, AGE]]
)
keep_ctrl  = downsample_healthy(ctrl_meta, AGE, SAMPLE_VARIABLE,
                                xdp_age_min, xdp_age_max, xdp_per_decade)
keep_nucseq = set(xdp_meta[SAMPLE_VARIABLE]) | keep_ctrl
adata_zone  = adata_zone[adata_zone.obs[SAMPLE_VARIABLE].isin(keep_nucseq)].copy()

# Downsample BICAN
bican_meta = bican_zone.obs.drop_duplicates(SAMPLE_VARIABLE)[[SAMPLE_VARIABLE, AGE]]
keep_bican = downsample_healthy(bican_meta, AGE, SAMPLE_VARIABLE,
                                xdp_age_min, xdp_age_max, xdp_per_decade)
bican_zone = bican_zone[bican_zone.obs[SAMPLE_VARIABLE].isin(keep_bican)].copy()

print(f"\nAfter matching — NucSeq: {adata_zone.obs[SAMPLE_VARIABLE].nunique()} donors "
      f"| BICAN: {bican_zone.obs[SAMPLE_VARIABLE].nunique()} donors")

plot_age_dist(adata_zone, bican_zone, title="Age distribution — after matching")

## Find de-repressed genes in BICAN (age-increasing silent genes)

In [ ]:
bican_cov = [c for c in ["sex", "pct_counts_mt"] if c in bican_zone.obs.columns]

results = find_derepressed_genes(
    bican_zone,
    age_col              = AGE,
    donor_col            = SAMPLE_VARIABLE,
    counts_col           = LIBRARY_SIZE_COL,
    other_covariate_cols = bican_cov,
    threshold            = 0.02,
    fdr                  = 0.05,
)

sig_genes = results[results["significant"]].index.tolist()
print(f"\nGene set size: {len(sig_genes)} genes")
results[results["significant"]].head(20)

## Merge datasets and store raw counts

In [ ]:
adata_zone.obs["dataset"] = "NucSeq"
bican_zone.obs["dataset"] = "BICAN"

full_adata = merge_adata([adata_zone, bican_zone])

# Ensure raw counts are in layers["counts"]
if "counts" not in full_adata.layers:
    full_adata.layers["counts"] = full_adata.X.copy()

print("Cells per dataset:")
print(full_adata.obs["dataset"].value_counts())
print("\nCells per condition:")
print(full_adata.obs[CONTRAST_VARIABLE].value_counts())

## Pseudobulk scoring (depth-robust)

**Why pseudobulk?**  
AUCell on single cells is sensitive to dropout — shallow cells score higher on low-expression gene sets just because they have more zeros. Summing raw counts per donor then normalising to CPM removes this artefact: every donor is on the same scale regardless of original library depth.

In [ ]:
donors = full_adata.obs[SAMPLE_VARIABLE].unique()
pb_raw = {}
for donor in donors:
    mask = full_adata.obs[SAMPLE_VARIABLE] == donor
    X    = full_adata[mask].layers["counts"]
    if scipy.sparse.issparse(X):
        X = X.toarray()
    pb_raw[donor] = X.sum(axis=0)

pb_df   = pd.DataFrame(pb_raw, index=full_adata.var_names).T   # donors × genes
pb_cpm  = pb_df.div(pb_df.sum(axis=1), axis=0) * 1e6           # CPM per donor
pb_log  = np.log1p(pb_cpm)                                     # log1p-CPM

# Score = mean log-CPM of de-repressed genes present in the data
available = [g for g in sig_genes if g in pb_log.columns]
missing   = set(sig_genes) - set(available)
if missing:
    print(f"Genes not found in merged data (skipped): {missing}")

pb_log["derepressed_score"] = pb_log[available].mean(axis=1)
print(f"Pseudobulk matrix: {pb_log.shape[0]} donors × {pb_log.shape[1]-1} genes")
print(f"Score computed from {len(available)} genes")

## Build per-donor summary table

In [ ]:
meta = (
    full_adata.obs
    .drop_duplicates(SAMPLE_VARIABLE)
    .set_index(SAMPLE_VARIABLE)
    [[AGE, CONTRAST_VARIABLE, LIBRARY_SIZE_COL, "dataset"]]
)

summary = pb_log[["derepressed_score"]].join(meta)
summary[AGE] = summary[AGE].astype(float)
summary["age_scaled"]    = (summary[AGE] - summary[AGE].mean()) / summary[AGE].std()
summary["is_xdp"]        = (summary[CONTRAST_VARIABLE] == CONTRAST_STIM).astype(int)
summary["log_total_umi"] = np.log10(summary[LIBRARY_SIZE_COL].astype(float))

print(summary.groupby(CONTRAST_VARIABLE)[["derepressed_score", LIBRARY_SIZE_COL]].describe().T)

## Sanity check: score vs UMI depth

With pseudobulk CPM the score should be **uncorrelated** with sequencing depth (unlike v1 where the correlation was strong).

In [ ]:
palette = {CONTRAST_BASELINE: "steelblue", CONTRAST_STIM: "tomato"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score vs raw UMI
sns.scatterplot(
    data=summary, x=LIBRARY_SIZE_COL, y="derepressed_score",
    hue=CONTRAST_VARIABLE, palette=palette, ax=axes[0]
)
axes[0].set_title("Score vs raw total UMI (should be uncorrelated)")
axes[0].set_xlabel("Total UMI (raw)")

# Score vs log UMI
sns.scatterplot(
    data=summary, x="log_total_umi", y="derepressed_score",
    hue=CONTRAST_VARIABLE, palette=palette, ax=axes[1]
)
axes[1].set_title("Score vs log10(total UMI)")
axes[1].set_xlabel("log10(total UMI)")

for ax in axes:
    ax.set_ylabel("De-repression score (mean log-CPM)")
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

# Pearson r per condition
for cond, grp in summary.groupby(CONTRAST_VARIABLE):
    r, p = __import__("scipy.stats", fromlist=["pearsonr"]).pearsonr(
        grp[LIBRARY_SIZE_COL].astype(float), grp["derepressed_score"]
    )
    print(f"{cond}: r={r:.3f}, p={p:.3f}")

## OLS: does de-repression depend on age, XDP status, or their interaction?

In [ ]:
# NucSeq only for the model (BICAN was the reference to define the gene set)
summary_nucseq = summary[summary["dataset"] == "NucSeq"].copy()

model = smf.ols(
    "derepressed_score ~ age_scaled * is_xdp",
    data=summary_nucseq,
).fit()
print(model.summary())

# Also fit with log_total_umi as a sanity-check covariate
# (should be non-significant if pseudobulk CPM did its job)
model_umi = smf.ols(
    "derepressed_score ~ age_scaled * is_xdp + log_total_umi",
    data=summary_nucseq,
).fit()
print("\n--- with log_total_umi covariate ---")
print(model_umi.summary())

## Per-donor boxplot by age bin and condition

In [ ]:
# Use cell-level de-repression for the boxplot:
# add pseudobulk score back to full_adata.obs for plotting per-donor distributions
score_map = pb_log["derepressed_score"].to_dict()
full_adata.obs["derepressed_score"] = full_adata.obs[SAMPLE_VARIABLE].map(score_map)

palette_cond = {CONTRAST_BASELINE: "steelblue", CONTRAST_STIM: "tomato"}

AGE_BIN_YEARS         = 5
GAP_WITHIN_BIN        = 0.6
GAP_BETWEEN_COND      = 0.3
GAP_BETWEEN_BINS      = 1.5

plot_df = full_adata.obs[full_adata.obs["dataset"] == "NucSeq"][
    [SAMPLE_VARIABLE, "derepressed_score", AGE, CONTRAST_VARIABLE]
].copy()
plot_df[AGE] = plot_df[AGE].astype(float)
plot_df["age_bin"] = (plot_df[AGE] // AGE_BIN_YEARS * AGE_BIN_YEARS).astype(int)

fig, ax = plt.subplots(figsize=(18, 5))
x_pos, x_ticks = 0, {}

for age_bin in sorted(plot_df["age_bin"].unique()):
    bin_df    = plot_df[plot_df["age_bin"] == age_bin]
    bin_start = x_pos
    for condition in [CONTRAST_BASELINE, CONTRAST_STIM]:
        cond_df = bin_df[bin_df[CONTRAST_VARIABLE] == condition]
        donors  = (
            cond_df.drop_duplicates(SAMPLE_VARIABLE)
            .sort_values(AGE)[SAMPLE_VARIABLE].tolist()
        )
        color = palette_cond[condition]
        for donor in donors:
            cells = cond_df[cond_df[SAMPLE_VARIABLE] == donor]["derepressed_score"].dropna()
            ax.boxplot(
                cells,
                positions=[x_pos], widths=0.5, patch_artist=True, showfliers=False,
                medianprops=dict(color="black", linewidth=1.5),
                boxprops=dict(facecolor=color, alpha=0.75),
                whiskerprops=dict(color=color), capprops=dict(color=color),
            )
            x_pos += GAP_WITHIN_BIN
        x_pos += GAP_BETWEEN_COND
    x_ticks[f"{age_bin}–{age_bin+AGE_BIN_YEARS}"] = (bin_start + x_pos - GAP_WITHIN_BIN) / 2
    x_pos += GAP_BETWEEN_BINS

ax.set_xticks(list(x_ticks.values()))
ax.set_xticklabels(list(x_ticks.keys()), fontsize=10)
ax.set_xlabel(f"Age ({AGE_BIN_YEARS}-year bins)", fontsize=12)
ax.set_ylabel("De-repression score (mean log-CPM)", fontsize=12)
ax.set_title("De-repression score per donor — pseudobulk CPM (v2)", fontsize=13)
legend_patches = [mpatches.Patch(color=c, alpha=0.75, label=l)
                  for l, c in palette_cond.items()]
ax.legend(handles=legend_patches, title="Condition", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()

## Score distribution per condition (violin)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.violinplot(
    data=summary_nucseq, x=CONTRAST_VARIABLE, y="derepressed_score",
    palette=palette_cond, inner="box", ax=ax
)
sns.stripplot(
    data=summary_nucseq, x=CONTRAST_VARIABLE, y="derepressed_score",
    color="black", alpha=0.5, size=4, ax=ax
)
ax.set_ylabel("De-repression score (mean log-CPM)")
ax.set_xlabel("")
ax.set_title("Overall score distribution (NucSeq donors)")
sns.despine()
plt.tight_layout()
plt.show()